# Day 10: End-to-End Orchestration, Observability & Verification

Complete E2E run of a synthetic patient from raw JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold readmission metric.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, current_timestamp, lit, expr
import duckdb
import os

warehouse_path = os.path.join(os.getcwd(), "spark-warehouse")

spark = SparkSession.builder \
    .appName("Day10_E2E") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created.")

In [ ]:
# 1. Mock Ingestion (Raw -> Bronze)
raw_data = [
    (1, "P999", "123-45-6789", "2023-10-01", "2023-10-05", 9201), # Index admission
    (2, "P999", "123-45-6789", "2023-10-15", "2023-10-18", 9201)  # 30-day readmission
]
raw_df = spark.createDataFrame(raw_data, ["encounter_id", "patient_id", "ssn", "start_date", "end_date", "concept_id"])
print("1. Raw Payload Received")
raw_df.show()

In [ ]:
# 2. Tokenization/De-identification
deid_df = raw_df.withColumn("ssn_hash", sha2(col("ssn"), 256)).drop("ssn")
print("2. De-identified (Tokenized)")
deid_df.show(truncate=False)

In [ ]:
# 3. Silver OMOP Transformation
silver_omop_df = deid_df.select(
    col("patient_id").alias("person_id"),
    col("encounter_id").alias("visit_occurrence_id"),
    col("concept_id").alias("visit_concept_id"),
    col("start_date").alias("visit_start_date"),
    col("end_date").alias("visit_end_date")
)
print("3. Transformed to OMOP (Silver)")
silver_omop_df.show()

In [ ]:
# 4. Data Quality (Mock Check)
dq_passed = silver_omop_df.filter(col("person_id").isNull()).count() == 0
print(f"4. Data Quality Check Passed: {dq_passed}")

In [ ]:
# 5. Gold Marts (DuckDB simulating Snowflake/dbt)
conn = duckdb.connect(':memory:')
conn.register('silver_visit', silver_omop_df.toPandas())

gold_query = """
WITH discharges AS (
    SELECT 
        person_id,
        visit_occurrence_id,
        visit_end_date as discharge_date
    FROM silver_visit
    WHERE visit_concept_id = 9201
),
readmissions AS (
    SELECT 
        d1.person_id,
        d1.visit_occurrence_id as index_visit_id,
        d2.visit_occurrence_id as readmission_visit_id
    FROM discharges d1
    JOIN silver_visit d2
        ON d1.person_id = d2.person_id
        AND d2.visit_start_date > d1.discharge_date
        AND datediff('day', cast(d1.discharge_date as date), cast(d2.visit_start_date as date)) <= 30
        AND d2.visit_concept_id = 9201
)
SELECT 
    v.person_id,
    COUNT(DISTINCT v.visit_occurrence_id) as total_index_admissions,
    COUNT(DISTINCT r.readmission_visit_id) as total_30d_readmissions
FROM silver_visit v
LEFT JOIN readmissions r ON v.person_id = r.person_id
WHERE v.visit_concept_id = 9201
GROUP BY v.person_id;
"""

print("5. Gold Mart: 30-Day Readmission Results")
print(conn.execute(gold_query).df())